# Notebook 05: Inference-Time Causal Validation (MoRF vs. LeRF) & RQ3 Answer
## Phase 5 of the Experimental Workflow (`results/final_experimental_report.md`)

**Causal Perturbation Protocol on Frozen Models (*Zero Re-Training*):**
1. **T5.1: Cumulative Field-Level MoRF vs. LeRF Curves ($m=1..5$)**:
   - **MoRF (*Most Relevant First*)**: progressively neutralizes the highest global-SHAP fields at inference time (steep performance drop expected).
   - **LeRF (*Least Relevant First*)**: neutralizes marginal least-relevant fields first (flatter curve expected).
   - Computes **Area Under the Degradation Curve (AUDC)** as a quantitative causal faithfulness metric.
2. **T5.2: Step $k=1$ Focus Analysis (Single Top-Field Occlusion)**:
   - Step $k=1$ on the MoRF curve corresponds by mathematical definition to single-field occlusion of each model's top-ranked header field.
   - Evaluates whether intrinsic **protocol header redundancy** cushions single-field ablation before severe degradation at $k \ge 2$, avoiding methodological retraining artifacts.
3. **T5.3: Computing $\Delta\text{Macro-F1}$, $\Delta\text{Balanced Accuracy}$, and Exporting `results/metrics/masking_results.csv`**.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, balanced_accuracy_score

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'
DATA_DIR = REPO_ROOT / 'data'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
from util.xai_utils import compute_audc
from xai.model_loader import load_checkpoint

seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Causal masking device: {device}")

### 5.1 Neutral Imputation Rule (`0.5` on valid packets only, preserving `0.0` padding)
In the CICIoMT2024 flow tensor, `0.0` encodes semantic zero padding for shorter flows. Overwriting features with `0.0` would mislead attention masks into treating packets as absent; active packets are instead neutralized with the **neutral median baseline `0.5`**.

In [ ]:
def mask_protocol_fields(x, field_indices, mask_val=0.5):
    """
    Mask specified header fields across all valid packets in the flow while preserving 0.0 padding.
    x: [B, 1, T, F]
    field_indices: list of header field indices [0..5]
    """
    x_masked = x.clone()
    for f in field_indices:
        # Identify valid packets (at least one non-zero field)
        valid_mask = (x_masked.abs().sum(dim=-1, keepdim=True) > 0)  # [B, 1, T, 1]
        # Mask only where valid_mask is True
        f_slice = x_masked[:, :, :, f:f+1]
        f_slice = torch.where(valid_mask, torch.tensor(mask_val, dtype=x.dtype, device=x.device), f_slice)
        x_masked[:, :, :, f:f+1] = f_slice
    return x_masked

print("[OK] Neutral imputation helper mask_protocol_fields ready.")

### 5.2 Loading Global Header Field Importance Rankings (SHAP)
Load per-field global importance ($F=6$) to establish the descending (**MoRF**) and ascending (**LeRF**) masking orders.

In [ ]:
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
npz_path = METRICS_DIR / 'xai_explanations.npz'

npz_data = np.load(npz_path, allow_pickle=True)  # trusted artifact generated by Notebook 03
shap_fields = {
    'Lopez17CNN': np.median(np.abs(npz_data['shap_lopez_field']), axis=0),
    'BiflowTransformer': np.median(np.abs(npz_data['shap_biflow_field']), axis=0),
    'MedDefenderTransformer': np.median(np.abs(npz_data['shap_meddef_field']), axis=0)
}

orders_morf = {}
orders_lerf = {}
for net, imp in shap_fields.items():
    orders_morf[net] = np.argsort(-imp)  # descending (Most Relevant First)
    orders_lerf[net] = np.argsort(imp)   # ascending (Least Relevant First)
    print(f"[{net}] MoRF order: {[FIELDS[i] for i in orders_morf[net]]}")
    print(f"[{net}] LeRF order: {[FIELDS[i] for i in orders_lerf[net]]}")

### 5.3 Evaluating MoRF vs. LeRF Curves on Frozen Models (Task T5.1 & T5.2)
Measure progressive classification degradation while neutralizing $m=1..5$ header fields at inference.

In [ ]:
from datasets.data_loader import get_loaders

_, _, tst_loader, _ = get_loaders(
    datasets=['cic-iomt'], num_tasks=1, nc_first_task=13, nc_incr_tasks=0,
    batch_size=128, num_workers=2, pin_memory=torch.cuda.is_available(),
    seed=0, num_pkts=10, fields=FIELDS
)
test_ldr = tst_loader[0]

RUN_IDS = {'Lopez17CNN': 'S2_R01_Lopez17CNN_multi_10pkts_seed0',
           'BiflowTransformer': 'S2_R02_BiflowTransformer_multi_10pkts_seed0',
           'MedDefenderTransformer': 'S2_R03_MedDefender_multi_10pkts_seed0'}
eval_models = {n: load_checkpoint(exp_name=f"cic-iomt_scratch_{e}", results_dir=str(RESULTS_DIR),
                                  network=n, device=str(device))[0] for n, e in RUN_IDS.items()}

degradation_data = {}
for net_name, model in eval_models.items():
    model.eval()
    curves = {'MoRF_F1': [], 'LeRF_F1': [], 'MoRF_BA': [], 'LeRF_BA': []}

    # Step m=0..5
    for m_step in range(6):
        fields_morf = orders_morf[net_name][:m_step].tolist()
        fields_lerf = orders_lerf[net_name][:m_step].tolist()

        yt, yp_m, yp_l = [], [], []
        with torch.no_grad():
            for bx, by in test_ldr:
                bx = bx.to(device)
                yt.extend(by.numpy())
                for fields_m, yp in ((fields_morf, yp_m), (fields_lerf, yp_l)):
                    out = model(mask_protocol_fields(bx, fields_m, mask_val=0.5))
                    out = out[0] if isinstance(out, (list, tuple)) else out
                    yp.extend(out.argmax(dim=-1).cpu().numpy())

        # Unrounded values: the drop at k=1 can be smaller than 0.1 points
        for tag, yp in (('MoRF', yp_m), ('LeRF', yp_l)):
            curves[f'{tag}_F1'].append(float(f1_score(yt, yp, average="macro") * 100))
            curves[f'{tag}_BA'].append(float(balanced_accuracy_score(yt, yp) * 100))

    degradation_data[net_name] = curves
    print(f"[{net_name}] MoRF F1: {np.round(curves['MoRF_F1'], 2)} | LeRF F1: {np.round(curves['LeRF_F1'], 2)}")

# Raw curves (Macro-F1 and Balanced Accuracy [%]) for Notebook 06
pd.DataFrame([{'Model': n, 'Curve': c, **{f'm={i}': v for i, v in enumerate(d[c])}}
              for n, d in degradation_data.items() for c in ('MoRF_F1', 'LeRF_F1', 'MoRF_BA', 'LeRF_BA')]
             ).to_csv(METRICS_DIR / 'morf_lerf_curves.csv', index=False)

TOP_FIELD = {net: FIELDS[orders_morf[net][0]] for net in degradation_data}

steps = np.array([0, 1, 2, 3, 4, 5])
audc_results = {}
for net, curves in degradation_data.items():
    audc_morf = compute_audc(steps, curves['MoRF_F1'])
    audc_lerf = compute_audc(steps, curves['LeRF_F1'])
    audc_results[net] = {'AUDC_MoRF': audc_morf, 'AUDC_LeRF': audc_lerf}
    print(f"[{net}] AUDC MoRF: {audc_morf:.2f} | AUDC LeRF: {audc_lerf:.2f} (Faithfulness Gap = {audc_lerf - audc_morf:.2f})")

### 5.4 Focused Step $k=1$ Analysis (Single Top-Field Occlusion)
At $k=1$, MoRF occludes only each model's top-ranked header field (verified in Section 5.2). A moderate drop indicates protocol field redundancy, whereas a sharp drop highlights single-field reliance.

In [ ]:
print("======================================================================")
print(" STEP k=1 ANALYSIS (SINGLE TOP-FIELD OCCLUSION)")
print("======================================================================")
for net, curves in degradation_data.items():
    base, k1 = curves['MoRF_F1'][0], curves['MoRF_F1'][1]
    d_ba = curves['MoRF_BA'][1] - curves['MoRF_BA'][0]
    print(f"[{net:24s}] Baseline: {base:.2f}% -> k=1 (without {TOP_FIELD[net]}): {k1:.2f}% "
          f"(Delta Macro-F1 = {k1 - base:+.2f}% | Delta Balanced Acc = {d_ba:+.2f}%)")

### 5.5 Exporting Causal Masking Results (`results/metrics/masking_results.csv`) (Task T5.3)
Export the causal perturbation summary table aligned with Section 3 of `results/final_experimental_report.md`.

In [ ]:
NETS = ['Lopez17CNN', 'BiflowTransformer', 'MedDefenderTransformer']

def step_cell(net, k):
    """Macro-F1 at MoRF step k along with Delta Macro-F1 and Delta Balanced Accuracy vs. m=0 [%]."""
    d = degradation_data[net]
    return (f"{d['MoRF_F1'][k]:.2f}% (Delta F1 {d['MoRF_F1'][k] - d['MoRF_F1'][0]:+.2f}%, "
            f"Delta BA {d['MoRF_BA'][k] - d['MoRF_BA'][0]:+.2f}%)")

steps_desc = [
    (1, 'MoRF Step k=1 (top field)',
     'Inference-time occlusion of top field (' + ', '.join(f"{n}: {f}" for n, f in TOP_FIELD.items()) + ')'),
    (2, 'MoRF Step k=2', 'Joint neutralization of top 2 header fields'),
    (3, 'MoRF Step k=3', 'Joint neutralization of top 3 header fields'),
    (5, 'MoRF Step k=5', 'Joint neutralization of 5 out of 6 header fields'),
]
table4_records = [{'Metric / Step': name, 'Description': desc, **{n: step_cell(n, k) for n in NETS}}
                  for k, name, desc in steps_desc]
table4_records += [
    {'Metric / Step': 'AUDC MoRF (\u2193)', 'Description': 'Area Under Degradation Curve (Most Relevant First)',
     **{n: f"{audc_results[n]['AUDC_MoRF']:.2f}" for n in NETS}},
    {'Metric / Step': 'AUDC LeRF (\u2191)', 'Description': 'Area Under Degradation Curve (Least Relevant First)',
     **{n: f"{audc_results[n]['AUDC_LeRF']:.2f}" for n in NETS}},
]

df_masking = pd.DataFrame(table4_records)
out_masking_csv = METRICS_DIR / 'masking_results.csv'
df_masking.to_csv(out_masking_csv, index=False)

print("\n======================================================================")
print(" CAUSAL VALIDATION AT INFERENCE SUMMARY (RQ3)")
print("======================================================================")
display(df_masking) if 'display' in globals() else print(df_masking.to_string(index=False))
print(f"\n[SUCCESS] Results saved to: {out_masking_csv}")

### 5.6 Generating Causal MoRF vs. LeRF & Step $k=1$ Figures (Task T5.3)
Render high-resolution 300 DPI multi-panel figures with explicit axis units:
1. **MoRF vs. LeRF Degradation Curves**: Macro-F1 [%] across neutralization steps $m \in [0..5]$.
2. **Step $k=1$ Detail (Top-Field Occlusion)**: baseline vs. single top-field ablation showing protocol sensitivity.
3. **Area Under Degradation Curve (AUDC)**: MoRF causal drop ($\downarrow$) vs. LeRF control baseline ($\uparrow$).

In [ ]:
# Scientific plotting configuration
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors_net = {'Lopez17CNN': '#d95f02', 'BiflowTransformer': '#7570b3', 'MedDefenderTransformer': '#1b9e77'}
steps_m = np.array([0, 1, 2, 3, 4, 5])

# 1. Progressive MoRF vs. LeRF Curves
ax1 = axes[0]
for net, curves in degradation_data.items():
    ax1.plot(steps_m, curves['MoRF_F1'], marker='o', linewidth=2.2, color=colors_net[net], label=f"{net} (MoRF)")
    ax1.plot(steps_m, curves['LeRF_F1'], marker='s', linestyle='--', linewidth=1.8, color=colors_net[net], alpha=0.65, label=f"{net} (LeRF)")

ax1.axvline(x=1, color='gray', linestyle=':', linewidth=1.5, label='Step k=1 (top field)')
ax1.set_title('A) Progressive MoRF vs. LeRF Curves (RQ3)', fontweight='bold', fontsize=12)
ax1.set_xlabel('Number of Neutralized Header Fields at Inference, m [0..5]')
ax1.set_ylabel('Macro F1-Score [%]')
ax1.set_xticks(steps_m)
ax1.set_xticklabels(['m=0\n(Intact)', 'm=1\n(top field)', 'm=2', 'm=3', 'm=4', 'm=5'])
ax1.set_ylim(0, 105)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', fontsize=8)

# 2. Step k=1 Detail (Single Top-Field Occlusion)
ax2 = axes[1]
nets = list(degradation_data.keys())
x_idx = np.arange(len(nets))
w_bar = 0.35

base_f1s = [degradation_data[n]['MoRF_F1'][0] for n in nets]
k1_f1s   = [degradation_data[n]['MoRF_F1'][1] for n in nets]

b1 = ax2.bar(x_idx - w_bar/2, base_f1s, width=w_bar, color='#4575b4', label='Intact Baseline (m=0)', edgecolor='black', linewidth=0.6)
b2 = ax2.bar(x_idx + w_bar/2, k1_f1s, width=w_bar, color='#fdae61', label='Single Top-Field Occlusion (k=1)', edgecolor='black', linewidth=0.6)

ax2.set_title('B) Macro-F1 Change at Step k=1 (Top Field)', fontweight='bold', fontsize=12)
ax2.set_xlabel('Evaluated Network Architecture')
ax2.set_ylabel('Macro F1-Score [%]')
ax2.set_xticks(x_idx)
ax2.set_xticklabels(nets, rotation=10)
ax2.grid(True, linestyle='--', alpha=0.5, axis='y')

for idx_n, n in enumerate(nets):
    delta = k1_f1s[idx_n] - base_f1s[idx_n]
    ax2.annotate(rf'$\Delta = {delta:+.1f}\%$', (x_idx[idx_n] + w_bar/2, k1_f1s[idx_n]), ha='center', va='bottom', xytext=(0, 3), textcoords='offset points', fontsize=9, fontweight='bold', color='#d73027')
ax2.legend(loc='lower right', fontsize=9)

# 3. AUDC Comparison (Area Under Degradation Curve)
ax3 = axes[2]
audc_morf_vals = [audc_results[n]['AUDC_MoRF'] for n in nets]
audc_lerf_vals = [audc_results[n]['AUDC_LeRF'] for n in nets]

b3 = ax3.bar(x_idx - w_bar/2, audc_morf_vals, width=w_bar, color='#d73027', label=r'AUDC MoRF ($\downarrow$ Causal Faithfulness)', edgecolor='black', linewidth=0.6)
b4 = ax3.bar(x_idx + w_bar/2, audc_lerf_vals, width=w_bar, color='#1a9850', label=r'AUDC LeRF ($\uparrow$ Contrast Control)', edgecolor='black', linewidth=0.6)

ax3.set_title('C) Area Under Degradation Curve: MoRF vs. LeRF', fontweight='bold', fontsize=12)
ax3.set_xlabel('Evaluated Network Architecture')
ax3.set_ylabel('Normalized AUDC [Percentage Points (p.p.)]')
ax3.set_xticks(x_idx)
ax3.set_xticklabels(nets, rotation=10)
ax3.grid(True, linestyle='--', alpha=0.5, axis='y')

for bar in b3 + b4:
    h = bar.get_height()
    ax3.annotate(f'{h:.1f}', (bar.get_x() + bar.get_width()/2, h), ha='center', va='bottom', xytext=(0, 3), textcoords='offset points', fontsize=9, fontweight='bold')
ax3.legend(loc='lower left', fontsize=9)

plt.tight_layout()
out_causal_png = FIGURES_DIR / 'causal_masking_comprehensive.png'
out_causal_pdf = FIGURES_DIR / 'causal_masking_comprehensive.pdf'
out_morf_png = FIGURES_DIR / 'morf_vs_lerf_curves.png'
out_morf_pdf = FIGURES_DIR / 'morf_vs_lerf_curves.pdf'

plt.savefig(out_causal_png, dpi=300, bbox_inches='tight')
plt.savefig(out_causal_pdf, bbox_inches='tight')
plt.savefig(out_morf_png, dpi=300, bbox_inches='tight')
plt.savefig(out_morf_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Causal validation figures saved to:\n  - {out_causal_png}\n  - {out_morf_png}')